# TCGA-BRCA Patient Treatment Profile V1 Review

This notebook reviews the saved TCGA-BRCA patient treatment profile v1 outputs from disk only.
It does not rerun the profile-building workflow, reread raw source files, freeze treatment arms,
normalize drug names, or perform modeling.


In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display


def detect_repo_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Unable to locate the repository root from the notebook path.')


def read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)


repo_root = detect_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'treatment-prep'
    / 'tcga_brca_patient_treatment_profile_v1_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest patient treatment profile v1 pointer not found: {latest_pointer_path}. '
        'Run script 20 first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
profile_path = repo_root / latest_pointer['patient_treatment_profile_v1_tsv']
conflict_path = repo_root / latest_pointer['patient_treatment_profile_v1_conflict_audit_tsv']
spec_path = repo_root / latest_pointer['patient_treatment_profile_v1_spec_tsv']
summary_path = repo_root / latest_pointer['patient_treatment_profile_v1_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']

for required_path in [profile_path, conflict_path, spec_path, summary_path, run_log_path]:
    if not required_path.exists():
        raise FileNotFoundError(f'Required patient treatment profile artifact not found: {required_path}')

profile_df = read_tsv(profile_path)
conflict_df = read_tsv(conflict_path)
spec_df = read_tsv(spec_path)
summary_df = read_tsv(summary_path)
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

if not bool(run_log.get('validation', {}).get('passed', False)):
    raise ValueError('run_log.json does not report validation.passed == true.')
if profile_df.empty:
    raise ValueError('patient_treatment_profile_v1.tsv contains no rows.')

results_root = (
    repo_root
    / '09-trials'
    / '01-tcga-only-source-audited'
    / '05-results'
)
results_root.mkdir(parents=True, exist_ok=True)

print(f"Run ID    : {latest_pointer['patient_treatment_profile_v1_run_id']}")
print(f"Overlap ID: {latest_pointer['treatment_os_overlap_v1_run_id']}")
print(f"OS ep ID  : {latest_pointer['os_endpoint_v1_run_id']}")
print(f"Pointer   : {latest_pointer_path}")


In [ ]:
# --- Write review tables to 05-results/ ---

review_profile_path = results_root / '112_patient_treatment_profile_v1.tsv'
review_conflict_path = results_root / '113_patient_treatment_profile_v1_conflict_audit.tsv'
review_spec_path = results_root / '114_patient_treatment_profile_v1_spec.tsv'
review_summary_path = results_root / '115_patient_treatment_profile_v1_summary.tsv'

profile_df.to_csv(review_profile_path, sep='\t', index=False)
conflict_df.to_csv(review_conflict_path, sep='\t', index=False)
spec_df.to_csv(review_spec_path, sep='\t', index=False)
summary_df.to_csv(review_summary_path, sep='\t', index=False)

print(f'Saved: {review_profile_path}')
print(f'Saved: {review_conflict_path}')
print(f'Saved: {review_spec_path}')
print(f'Saved: {review_summary_path}')


In [ ]:
# --- Pointer, validation, and key counts ---

print('=== Latest pointer ===')
display(pd.DataFrame([latest_pointer]))

validation = run_log.get('validation', {})
print('\n=== Validation ===')
display(
    pd.DataFrame([
        {'check': k, 'value': str(v)}
        for k, v in validation.items()
    ])
)

print('\n=== Key counts ===')
display(
    pd.DataFrame([
        {'metric': k, 'value': str(v)}
        for k, v in run_log.get('counts', {}).items()
    ])
)

print('\n=== Saved summary TSV ===')
display(summary_df)


In [ ]:
# --- Treatment profile structure ---

status_distribution_df = (
    profile_df
    .groupby(['treatment_profile_status', 'treatment_profile_requires_manual_review'], as_index=False)
    .size()
    .rename(columns={'size': 'patient_count'})
    .sort_values(['patient_count', 'treatment_profile_status'], ascending=[False, True])
    .reset_index(drop=True)
)

dominant_identifiable_df = pd.DataFrame(
    [
        {
            'group': 'dominant_therapy_type_identifiable',
            'patient_count': int((profile_df['dominant_therapy_type_if_any'] != '').sum()),
        },
        {
            'group': 'dominant_therapy_type_blank',
            'patient_count': int((profile_df['dominant_therapy_type_if_any'] == '').sum()),
        },
    ]
)

regimen_context_df = (
    profile_df
    .groupby(['has_any_regimen_context', 'regimen_context_single_or_mixed'], as_index=False)
    .size()
    .rename(columns={'size': 'patient_count'})
    .sort_values(['patient_count', 'regimen_context_single_or_mixed'], ascending=[False, True])
    .reset_index(drop=True)
)

timing_coverage_df = (
    profile_df['has_any_treatment_timing']
    .value_counts()
    .rename_axis('has_any_treatment_timing')
    .reset_index(name='patient_count')
    .sort_values('has_any_treatment_timing')
    .reset_index(drop=True)
)

drug_timing_inverted_df = profile_df.loc[
    profile_df['treatment_profile_flags_json'].str.contains('drug_timing_window_aggregated_inverted', regex=False),
    [
        'bcr_patient_barcode',
        'drug_row_count',
        'earliest_drug_start_days',
        'latest_drug_end_days',
        'treatment_profile_flags_json',
    ],
].reset_index(drop=True)

print('=== Treatment profile status distribution ===')
display(status_distribution_df)

print('\n=== Dominant-type identifiability ===')
display(dominant_identifiable_df)

print('\n=== Regimen-context coverage ===')
display(regimen_context_df)

print('\n=== Timing coverage ===')
display(timing_coverage_df)

print('\n=== Drug timing inverted flags ===')
display(drug_timing_inverted_df)


In [ ]:
# --- Conflict audit and saved outputs preview ---

conflict_counts_df = (
    conflict_df
    .groupby(['conflict_type', 'review_priority'], as_index=False)
    .size()
    .rename(columns={'size': 'patient_count'})
    .sort_values(['review_priority', 'patient_count', 'conflict_type'], ascending=[True, False, True])
    .reset_index(drop=True)
)

print('=== Conflict counts ===')
display(conflict_counts_df)

print('\n=== Conflict audit (first 20 rows) ===')
display(conflict_df.head(20))

print('\n=== Patient treatment profile (first 20 rows) ===')
display(profile_df.head(20))

print('\n=== Spec ===')
display(spec_df)
